In [ ]:
import pandas as pd
import yaml
import getpass
from urllib.parse import quote_plus
from sqlalchemy import create_engine

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

In [2]:
config['input_data']

{'application': '../data/raw/application_train.csv',
 'bureau': '../data/raw/bureau.csv',
 'previous_application': '../data/raw/previous_application.csv'}

In [3]:
bureau = pd.read_csv(config['input_data']['bureau'])
bureau.shape

(1716428, 17)

In [4]:
bureau_columns = [
    'SK_ID_CURR',
    'CREDIT_DAY_OVERDUE',
    'AMT_CREDIT_SUM_OVERDUE'
]

bureau_trimmed_df = bureau[bureau_columns].copy()
bureau_trimmed_df.shape

(1716428, 3)

In [5]:
bureau_trimmed_df.head()

,SK_ID_CURR,CREDIT_DAY_OVERDUE,AMT_CREDIT_SUM_OVERDUE
0,215354,0,0.0
1,215354,0,0.0
2,215354,0,0.0
3,215354,0,0.0
4,215354,0,0.0


In [6]:
bureau_trimmed_df.isnull().sum()

SK_ID_CURR                0
CREDIT_DAY_OVERDUE        0
AMT_CREDIT_SUM_OVERDUE    0
dtype: int64

In [7]:
bureau_trimmed_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1716428 entries, 0 to 1716427
Data columns (total 3 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_CURR              int64  
 1   CREDIT_DAY_OVERDUE      int64  
 2   AMT_CREDIT_SUM_OVERDUE  float64
dtypes: float64(1), int64(2)
memory usage: 39.3 MB


In [8]:
bureau_trimmed_df.describe()

,SK_ID_CURR,CREDIT_DAY_OVERDUE,AMT_CREDIT_SUM_OVERDUE
count,1.716428e+06,1.716428e+06,1.716428e+06
mean,2.782149e+05,8.181666e-01,3.791276e+01
std,1.029386e+05,3.654443e+01,5.937650e+03
min,1.000010e+05,0.000000e+00,0.000000e+00
25%,1.888668e+05,0.000000e+00,0.000000e+00
50%,2.780550e+05,0.000000e+00,0.000000e+00
75%,3.674260e+05,0.000000e+00,0.000000e+00
max,4.562550e+05,2.792000e+03,3.756681e+06


In [9]:
bureau_trimmed_df[
    ['CREDIT_DAY_OVERDUE', 'AMT_CREDIT_SUM_OVERDUE']
].lt(0).sum()

CREDIT_DAY_OVERDUE        0
AMT_CREDIT_SUM_OVERDUE    0
dtype: int64

In [10]:
bureau_trimmed_df.to_csv(
    config['output_data']['bureau'],
    index=False
)

In [11]:
pd.read_csv(config['output_data']['bureau']).head()

,SK_ID_CURR,CREDIT_DAY_OVERDUE,AMT_CREDIT_SUM_OVERDUE
0,215354,0,0.0
1,215354,0,0.0
2,215354,0,0.0
3,215354,0,0.0
4,215354,0,0.0


> **Aroa's fix:** dropped a duplicate save cell pointing at a mismatched
> filename, and added the MySQL load step below to match the `bureau`
> table schema -- including a filter for rows whose `SK_ID_CURR` isn't in
> `application` (the FK would otherwise reject them).

## Load into MySQL

In [ ]:
db = config['database']
password = getpass.getpass("MySQL password (press Enter if none): ")
engine = create_engine(f"mysql+pymysql://{db['user']}:{quote_plus(password)}@{db['host']}/{db['name']}")

# bureau.SK_ID_CURR has a foreign key to application.SK_ID_CURR -- bureau.csv
# includes applicants outside application_train.csv (Kaggle's test set), so
# some rows would violate that FK. Keep only rows whose SK_ID_CURR is
# already loaded in the application table.
existing_ids = pd.read_sql("SELECT SK_ID_CURR FROM application", engine)["SK_ID_CURR"]
bureau_trimmed_df = bureau_trimmed_df[bureau_trimmed_df["SK_ID_CURR"].isin(existing_ids)]
bureau_trimmed_df.shape

In [13]:
bureau_trimmed_df.to_sql(
    "bureau",
    engine,
    if_exists="append",
    index=False,
    chunksize=10000,
    method="multi",
)

1465248